In [ ]:
# ============================================================
# CHUNK 1: LOAD DATA + DEFINE FINAL VARIABLES
# ============================================================

!pip -q install imbalanced-learn openpyxl

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from google.colab import files

# -----------------------------
# Upload development file
# -----------------------------
uploaded = files.upload()

DEV_FILE = "CustomerChurn_Data.csv"
df = pd.read_csv(DEV_FILE)

print("Development file shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# -----------------------------
# Core settings
# -----------------------------
TARGET = "CHURN"
ALT_TARGET = "CHURNDEP"
SPLIT_COL = "CALIBRAT"

# -----------------------------
# Sanity check: compare CHURNDEP to CHURN where both exist
# -----------------------------
if ALT_TARGET in df.columns:
    compare_df = df[[TARGET, ALT_TARGET]].dropna()
    if len(compare_df) > 0:
        agreement = (compare_df[TARGET] == compare_df[ALT_TARGET]).mean()
        print(f"\nAgreement between CHURN and CHURNDEP where both exist: {agreement:.4f}")
    else:
        print("\nNo overlapping non-missing rows between CHURN and CHURNDEP to compare.")

# -----------------------------
# Keep only defensible base variables
# -----------------------------
base_vars = [
    "REVENUE",
    "RECCHRGE",
    "OVERAGE",
    "ROAM",
    "DROPVCE",
    "BLCKVCE",
    "UNANSVCE",
    "DROPBLK",
    "CUSTCARE",
    "OUTCALLS",
    "INCALLS",
    "MOU",
    "MONTHS"
]

existing_base_vars = [c for c in base_vars if c in df.columns]
missing_base_vars = [c for c in base_vars if c not in df.columns]

print("\nBase variables found:", existing_base_vars)
if missing_base_vars:
    print("Base variables missing:", missing_base_vars)

# -----------------------------
# Engineer only 4 features
# -----------------------------
def engineer_final_features(data):
    data = data.copy()

    # 1) log_Usage_Ratio
    if {"MOU", "REVENUE"}.issubset(data.columns):
        usage_ratio = data["MOU"] / data["REVENUE"].replace(0, np.nan)
        usage_ratio = usage_ratio.replace([np.inf, -np.inf], np.nan)
        data["log_Usage_Ratio"] = np.log1p(usage_ratio.clip(lower=0))

    # 2) Service_Issue_Total
    if {"DROPVCE", "BLCKVCE", "UNANSVCE"}.issubset(data.columns):
        data["Service_Issue_Total"] = data["DROPVCE"] + data["BLCKVCE"] + data["UNANSVCE"]

    # 3) CALL_FAILURE_RATE
    if {"DROPBLK", "OUTCALLS", "INCALLS"}.issubset(data.columns):
        data["CALL_FAILURE_RATE"] = data["DROPBLK"] / (data["OUTCALLS"] + data["INCALLS"] + 1)

    # 4) CUSTCARE_PER_MONTH
    if {"CUSTCARE", "MONTHS"}.issubset(data.columns):
        data["CUSTCARE_PER_MONTH"] = data["CUSTCARE"] / (data["MONTHS"] + 1)

    return data

df = engineer_final_features(df)

engineered_vars = [
    "log_Usage_Ratio",
    "Service_Issue_Total",
    "CALL_FAILURE_RATE",
    "CUSTCARE_PER_MONTH"
]

existing_engineered_vars = [c for c in engineered_vars if c in df.columns]
print("\nEngineered variables created:", existing_engineered_vars)

# -----------------------------
# Final feature list
# -----------------------------
feature_cols = existing_base_vars + existing_engineered_vars

# Remove duplicates just in case
feature_cols = list(dict.fromkeys(feature_cols))

print("\nFinal feature list:")
print(feature_cols)
print(f"\nTotal features before preprocessing: {len(feature_cols)}")

# -----------------------------
# Split into calibration/train and validation
# -----------------------------
train_df = df[df[SPLIT_COL] == 1].copy()
valid_df = df[df[SPLIT_COL] == 0].copy()

X_train = train_df[feature_cols].copy()
y_train = train_df[TARGET].astype(int)

X_valid = valid_df[feature_cols].copy()
y_valid = valid_df[TARGET].astype(int)

print("\nTrain shape:", X_train.shape)
print("Validation shape:", X_valid.shape)

print("\nChurn rates:")
print("Train:", y_train.mean())
print("Validation:", y_valid.mean())

print("\nMissingness in final features:")
print(X_train.isna().mean().sort_values(ascending=False))

Saving CustomerChurn_Data.csv to CustomerChurn_Data.csv
Development file shape: (850, 84)

Columns:
['REVENUE', 'MOU', 'RECCHRGE', 'DIRECTAS', 'OVERAGE', 'ROAM', 'CHANGEM', 'CHANGER', 'DROPVCE', 'BLCKVCE', 'UNANSVCE', 'CUSTCARE', 'THREEWAY', 'MOUREC', 'OUTCALLS', 'INCALLS', 'PEAKVCE', 'OPEAKVCE', 'DROPBLK', 'CALLFWDV', 'CALLWAIT', 'CHURN', 'MONTHS', 'UNIQSUBS', 'ACTVSUBS', 'CSA', 'PHONES', 'MODELS', 'EQPDAYS', 'CUSTOMER', 'AGE1', 'AGE2', 'CHILDREN', 'CREDITA', 'CREDITAA', 'CREDITB', 'CREDITC', 'CREDITDE', 'CREDITGY', 'CREDITZ', 'CREDIT_RATING', 'PRIZMRUR', 'PRIZMUB', 'PRIZMTWN', 'Column 45', 'REFURB', 'WEBCAP', 'TRUCK', 'RV', 'OCCPROF', 'OCCCLER', 'OCCCRFT', 'OCCSTUD', 'OCCHMKR', 'OCCRET', 'OCCSELF', 'OCC', 'OCC_LABEL', 'OWNRENT', 'MARRYUN', 'MARRYYES', 'MARRYNO', 'MARRY', 'MARRY_LABEL', 'MAILORD', 'MAILRES', 'MAILFLAG', 'TRAVEL', 'PCOWN', 'CREDITCD', 'RETCALLS', 'RETACCPT', 'NEWCELLY', 'NEWCELLN', 'REFER', 'INCMISS', 'INCOME', 'MCYCLE', 'CREDITAD', 'SETPRCM', 'SETPRC', 'RETCALL', 'CAL

In [ ]:
# ============================================================
# CHUNK 2: BUILD FINAL THREE MODELS
# ============================================================

from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from imblearn.over_sampling import SMOTE

# -----------------------------
# Numeric-only pipeline
# All final selected variables are numeric
# -----------------------------
preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# -----------------------------
# Final models
# -----------------------------
final_models = {
    "Logistic_L2": Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(
            max_iter=5000,
            solver="liblinear",
            C=1.0
        ))
    ]),

    "Weighted_Logistic": Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(
            max_iter=5000,
            solver="liblinear",
            C=1.0,
            class_weight="balanced"
        ))
    ]),

    "Reduced_Logistic_L1_Balanced": Pipeline([
        ("prep", preprocessor),
        ("select", SelectFromModel(
            LogisticRegression(
                penalty="l1",
                solver="liblinear",
                max_iter=5000,
                C=0.5,
                class_weight="balanced"
            ),
            threshold="median"
        )),
        ("model", LogisticRegression(
            max_iter=5000,
            solver="liblinear",
            C=1.0,
            class_weight="balanced"
        ))
    ])
}

# -----------------------------
# Fit models
# -----------------------------
fitted_models = {}

for name, model in final_models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    print(f"Fitted: {name}")

Fitted: Logistic_L2
Fitted: Weighted_Logistic
Fitted: Reduced_Logistic_L1_Balanced


In [ ]:
# ============================================================
# CHUNK 3: VALIDATION SCORING + THRESHOLD TUNING
# ============================================================

from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, balanced_accuracy_score

def score_model(model, X, y, threshold=0.5):
    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y, preds).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    f1 = (2 * precision * sensitivity / (precision + sensitivity)
          if pd.notna(precision) and pd.notna(sensitivity) and (precision + sensitivity) > 0
          else np.nan)

    return {
        "AUC": roc_auc_score(y, probs),
        "Accuracy": accuracy_score(y, preds),
        "Balanced_Accuracy": balanced_accuracy_score(y, preds),
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "Precision": precision,
        "F1": f1,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn
    }

# -----------------------------
# Default validation results
# -----------------------------
validation_results = []

for name, model in fitted_models.items():
    metrics = score_model(model, X_valid, y_valid, threshold=0.5)
    metrics["Model"] = name
    metrics["Threshold"] = 0.50
    validation_results.append(metrics)

validation_results_df = pd.DataFrame(validation_results)

print("Validation results at threshold 0.50:")
display(validation_results_df.sort_values("AUC", ascending=False))

# -----------------------------
# Threshold tuning
# -----------------------------
def threshold_table(model, X, y, model_name, thresholds=np.arange(0.10, 0.91, 0.05)):
    probs = model.predict_proba(X)[:, 1]
    rows = []

    for t in thresholds:
        preds = (probs >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, preds).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        f1 = (2 * precision * sensitivity / (precision + sensitivity)
              if pd.notna(precision) and pd.notna(sensitivity) and (precision + sensitivity) > 0
              else np.nan)

        rows.append({
            "Model": model_name,
            "Threshold": round(float(t), 2),
            "AUC": roc_auc_score(y, probs),
            "Accuracy": accuracy_score(y, preds),
            "Balanced_Accuracy": balanced_accuracy_score(y, preds),
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "Precision": precision,
            "F1": f1,
            "TP": tp,
            "FP": fp,
            "TN": tn,
            "FN": fn
        })

    return pd.DataFrame(rows)

threshold_results = []

for name, model in fitted_models.items():
    temp = threshold_table(model, X_valid, y_valid, name)
    threshold_results.append(temp)

threshold_results_df = pd.concat(threshold_results, ignore_index=True)

print("\nBest threshold candidates on validation:")
display(
    threshold_results_df.sort_values(
        ["Balanced_Accuracy", "AUC", "Sensitivity"],
        ascending=False
    ).head(20)
)

validation_results_df.to_csv("validation_results_final.csv", index=False)
threshold_results_df.to_csv("threshold_tuning_final.csv", index=False)

Validation results at threshold 0.50:


,AUC,Accuracy,Balanced_Accuracy,Sensitivity,Specificity,Precision,F1,TP,FP,TN,FN,Model,Threshold
2,0.563045,0.522013,0.551471,0.583333,0.519608,0.045455,0.084337,7,147,159,5,Reduced_Logistic_L1_Balanced,0.5
0,0.562364,0.584906,0.544118,0.500000,0.588235,0.045455,0.083333,6,126,180,6,Logistic_L2,0.5
1,0.560730,0.518868,0.549837,0.583333,0.516340,0.045161,0.083832,7,148,158,5,Weighted_Logistic,0.5



Best threshold candidates on validation:


,Model,Threshold,AUC,Accuracy,Balanced_Accuracy,Sensitivity,Specificity,Precision,F1,TP,FP,TN,FN
7,Logistic_L2,0.45,0.562364,0.446541,0.592320,0.750000,0.434641,0.049451,0.092784,9,173,133,3
23,Weighted_Logistic,0.40,0.560730,0.238994,0.564542,0.916667,0.212418,0.043651,0.083333,11,241,65,1
45,Reduced_Logistic_L1_Balanced,0.65,0.563045,0.911950,0.553922,0.166667,0.941176,0.100000,0.125000,2,18,288,10
42,Reduced_Logistic_L1_Balanced,0.50,0.563045,0.522013,0.551471,0.583333,0.519608,0.045455,0.084337,7,147,159,5
24,Weighted_Logistic,0.45,0.560730,0.367925,0.551471,0.750000,0.352941,0.043478,0.082192,9,198,108,3
25,Weighted_Logistic,0.50,0.560730,0.518868,0.549837,0.583333,0.516340,0.045161,0.083832,7,148,158,5
6,Logistic_L2,0.40,0.562364,0.283019,0.547386,0.833333,0.261438,0.042373,0.080645,10,226,80,2
41,Reduced_Logistic_L1_Balanced,0.45,0.563045,0.355346,0.544935,0.750000,0.339869,0.042654,0.080717,9,202,104,3
8,Logistic_L2,0.50,0.562364,0.584906,0.544118,0.500000,0.588235,0.045455,0.083333,6,126,180,6
28,Weighted_Logistic,0.65,0.560730,0.889937,0.542484,0.166667,0.918301,0.074074,0.102564,2,25,281,10


In [ ]:
# ============================================================
# CHUNK 4: OPTIONAL EXPLORATORY SMOTE MODEL
# ============================================================

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE

# -----------------------------
# Manual preprocessing for SMOTE
# -----------------------------
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_imp = imputer.fit_transform(X_train)
X_valid_imp = imputer.transform(X_valid)

X_train_scaled = scaler.fit_transform(X_train_imp)
X_valid_scaled = scaler.transform(X_valid_imp)

# -----------------------------
# Apply SMOTE on TRAIN only
# -----------------------------
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

# -----------------------------
# Fit logistic regression
# -----------------------------
smote_model = LogisticRegression(
    max_iter=5000,
    solver="liblinear",
    C=1.0
)

smote_model.fit(X_train_smote, y_train_smote)

# Store everything together so we can score later
fitted_models["SMOTE_Logistic_Exploratory"] = {
    "model": smote_model,
    "imputer": imputer,
    "scaler": scaler
}

# -----------------------------
# Score on validation
# -----------------------------
valid_probs = smote_model.predict_proba(X_valid_scaled)[:, 1]
valid_preds = (valid_probs >= 0.5).astype(int)

from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, balanced_accuracy_score

tn, fp, fn, tp = confusion_matrix(y_valid, valid_preds).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
f1 = (2 * precision * sensitivity / (precision + sensitivity)
      if pd.notna(precision) and pd.notna(sensitivity) and (precision + sensitivity) > 0
      else np.nan)

smote_validation = {
    "Model": "SMOTE_Logistic_Exploratory",
    "Threshold": 0.50,
    "AUC": roc_auc_score(y_valid, valid_probs),
    "Accuracy": accuracy_score(y_valid, valid_preds),
    "Balanced_Accuracy": balanced_accuracy_score(y_valid, valid_preds),
    "Sensitivity": sensitivity,
    "Specificity": specificity,
    "Precision": precision,
    "F1": f1,
    "TP": tp,
    "FP": fp,
    "TN": tn,
    "FN": fn
}

smote_validation_df = pd.DataFrame([smote_validation])

print("Exploratory SMOTE validation result:")
display(smote_validation_df)

smote_validation_df.to_csv("validation_smote_exploratory.csv", index=False)

Exploratory SMOTE validation result:


,Model,Threshold,AUC,Accuracy,Balanced_Accuracy,Sensitivity,Specificity,Precision,F1,TP,FP,TN,FN
0,SMOTE_Logistic_Exploratory,0.5,0.552015,0.518868,0.589869,0.666667,0.513072,0.050955,0.094675,8,149,157,4


In [ ]:
# ============================================================
# CHUNK 5: LOAD VERIFICATION DATA + SCORE MODELS
# ============================================================

from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, balanced_accuracy_score

# -----------------------------
# Upload verification file
# -----------------------------
uploaded_verify = files.upload()

VERIFY_FILE = "CustomerChurn_Verify.csv"
vf = pd.read_csv(VERIFY_FILE)

print("Verification file shape:", vf.shape)

# -----------------------------
# Apply same feature engineering
# -----------------------------
vf = engineer_final_features(vf)

# -----------------------------
# Build verification matrix
# Use the exact same feature list as training
# -----------------------------
for col in feature_cols:
    if col not in vf.columns:
        vf[col] = np.nan

X_verify = vf[feature_cols].copy()
y_verify = vf[TARGET].astype(int)

print("\nVerification churn rate:", y_verify.mean())
print("\nMissingness in verification features:")
print(X_verify.isna().mean().sort_values(ascending=False))

# -----------------------------
# Score all fitted models
# Handles both normal pipeline models
# and the manually-preprocessed SMOTE model
# -----------------------------
verify_results = []

for name, model_obj in fitted_models.items():

    if name == "SMOTE_Logistic_Exploratory":
        imputer = model_obj["imputer"]
        scaler = model_obj["scaler"]
        model = model_obj["model"]

        X_verify_imp = imputer.transform(X_verify)
        X_verify_scaled = scaler.transform(X_verify_imp)

        probs = model.predict_proba(X_verify_scaled)[:, 1]
        preds = (probs >= 0.5).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_verify, preds).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        f1 = (2 * precision * sensitivity / (precision + sensitivity)
              if pd.notna(precision) and pd.notna(sensitivity) and (precision + sensitivity) > 0
              else np.nan)

        metrics = {
            "AUC": roc_auc_score(y_verify, probs),
            "Accuracy": accuracy_score(y_verify, preds),
            "Balanced_Accuracy": balanced_accuracy_score(y_verify, preds),
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "Precision": precision,
            "F1": f1,
            "TP": tp,
            "FP": fp,
            "TN": tn,
            "FN": fn
        }

    else:
        metrics = score_model(model_obj, X_verify, y_verify, threshold=0.5)

    metrics["Model"] = name
    metrics["Threshold"] = 0.50
    verify_results.append(metrics)

verify_results_df = pd.DataFrame(verify_results)

print("\nVerification results at threshold 0.50:")
display(verify_results_df.sort_values("AUC", ascending=False))

verify_results_df.to_csv("verification_results_final.csv", index=False)
print("\nSaved: verification_results_final.csv")

Saving CustomerChurn_Verify.csv to CustomerChurn_Verify.csv
Verification file shape: (150, 84)

Verification churn rate: 0.20666666666666667

Missingness in verification features:
REVENUE                0.0
RECCHRGE               0.0
OVERAGE                0.0
ROAM                   0.0
DROPVCE                0.0
BLCKVCE                0.0
UNANSVCE               0.0
DROPBLK                0.0
CUSTCARE               0.0
OUTCALLS               0.0
INCALLS                0.0
MOU                    0.0
MONTHS                 0.0
log_Usage_Ratio        0.0
Service_Issue_Total    0.0
CALL_FAILURE_RATE      0.0
CUSTCARE_PER_MONTH     0.0
dtype: float64

Verification results at threshold 0.50:


,AUC,Accuracy,Balanced_Accuracy,Sensitivity,Specificity,Precision,F1,TP,FP,TN,FN,Model,Threshold
2,0.655462,0.533333,0.586609,0.677419,0.495798,0.259259,0.375000,21,60,59,10,Reduced_Logistic_L1_Balanced,0.5
0,0.625644,0.633333,0.613852,0.580645,0.647059,0.300000,0.395604,18,42,77,13,Logistic_L2,0.5
1,0.625644,0.573333,0.611819,0.677419,0.546218,0.280000,0.396226,21,54,65,10,Weighted_Logistic,0.5
3,0.623204,0.566667,0.583763,0.612903,0.554622,0.263889,0.368932,19,53,66,12,SMOTE_Logistic_Exploratory,0.5



Saved: verification_results_final.csv


In [ ]:
# ============================================================
# CHUNK 6: FINAL COMPARISON TABLES + REDUCED FEATURE COUNT
# ============================================================

# -----------------------------
# Validation table
# -----------------------------
val_table = validation_results_df.copy()
val_table["Dataset"] = "Validation"

# If SMOTE exploratory was run, include it in validation results
if "smote_validation_df" in globals():
    smote_val_table = smote_validation_df.copy()
    smote_val_table["Dataset"] = "Validation"
    val_table = pd.concat([val_table, smote_val_table], ignore_index=True)

# -----------------------------
# Verification table
# -----------------------------
ver_table = verify_results_df.copy()
ver_table["Dataset"] = "Verification"

# -----------------------------
# Long comparison table
# -----------------------------
common_cols = [
    "Model", "Dataset", "Threshold", "AUC", "Accuracy", "Balanced_Accuracy",
    "Sensitivity", "Specificity", "Precision", "F1", "TP", "FP", "TN", "FN"
]

final_long_table = pd.concat(
    [val_table[common_cols], ver_table[common_cols]],
    ignore_index=True
)

print("Final long comparison table:")
display(final_long_table.sort_values(["Dataset", "AUC"], ascending=[True, False]))

# -----------------------------
# Wide comparison table
# -----------------------------
metric_cols = [
    "AUC", "Accuracy", "Balanced_Accuracy",
    "Sensitivity", "Specificity", "Precision", "F1"
]

val_wide = val_table[["Model"] + metric_cols].copy()
val_wide.columns = ["Model"] + [f"Validation_{c}" for c in metric_cols]

ver_wide = ver_table[["Model"] + metric_cols].copy()
ver_wide.columns = ["Model"] + [f"Verification_{c}" for c in metric_cols]

final_wide_table = val_wide.merge(ver_wide, on="Model", how="outer")

print("\nFinal wide comparison table:")
display(final_wide_table.sort_values("Verification_AUC", ascending=False))

# -----------------------------
# Reduced feature count
# -----------------------------
reduced_model = fitted_models["Reduced_Logistic_L1_Balanced"]
selector = reduced_model.named_steps["select"]
selected_mask = selector.get_support()

selected_feature_count = int(selected_mask.sum())

print("\nOriginal feature count before reduction:", len(feature_cols))
print("Selected feature count after reduction:", selected_feature_count)

# -----------------------------
# Optional: show selected feature names
# Since all final features are numeric and scaling does not change count/order,
# selected names map directly to feature_cols
# -----------------------------
selected_feature_names = [feature_cols[i] for i, keep in enumerate(selected_mask) if keep]

print("\nSelected features in Reduced Logistic model:")
for f in selected_feature_names:
    print("-", f)

selected_features_df = pd.DataFrame({
    "Selected_Feature": selected_feature_names
})

# -----------------------------
# Save outputs
# -----------------------------
final_long_table.to_csv("final_comparison_long.csv", index=False)
final_wide_table.to_csv("final_comparison_wide.csv", index=False)
selected_features_df.to_csv("reduced_model_selected_features.csv", index=False)

print("\nSaved:")
print("- final_comparison_long.csv")
print("- final_comparison_wide.csv")
print("- reduced_model_selected_features.csv")

Final long comparison table:


,Model,Dataset,Threshold,AUC,Accuracy,Balanced_Accuracy,Sensitivity,Specificity,Precision,F1,TP,FP,TN,FN
2,Reduced_Logistic_L1_Balanced,Validation,0.5,0.563045,0.522013,0.551471,0.583333,0.519608,0.045455,0.084337,7,147,159,5
0,Logistic_L2,Validation,0.5,0.562364,0.584906,0.544118,0.500000,0.588235,0.045455,0.083333,6,126,180,6
1,Weighted_Logistic,Validation,0.5,0.560730,0.518868,0.549837,0.583333,0.516340,0.045161,0.083832,7,148,158,5
3,SMOTE_Logistic_Exploratory,Validation,0.5,0.552015,0.518868,0.589869,0.666667,0.513072,0.050955,0.094675,8,149,157,4
6,Reduced_Logistic_L1_Balanced,Verification,0.5,0.655462,0.533333,0.586609,0.677419,0.495798,0.259259,0.375000,21,60,59,10
4,Logistic_L2,Verification,0.5,0.625644,0.633333,0.613852,0.580645,0.647059,0.300000,0.395604,18,42,77,13
5,Weighted_Logistic,Verification,0.5,0.625644,0.573333,0.611819,0.677419,0.546218,0.280000,0.396226,21,54,65,10
7,SMOTE_Logistic_Exploratory,Verification,0.5,0.623204,0.566667,0.583763,0.612903,0.554622,0.263889,0.368932,19,53,66,12



Final wide comparison table:


,Model,Validation_AUC,Validation_Accuracy,Validation_Balanced_Accuracy,Validation_Sensitivity,Validation_Specificity,Validation_Precision,Validation_F1,Verification_AUC,Verification_Accuracy,Verification_Balanced_Accuracy,Verification_Sensitivity,Verification_Specificity,Verification_Precision,Verification_F1
1,Reduced_Logistic_L1_Balanced,0.563045,0.522013,0.551471,0.583333,0.519608,0.045455,0.084337,0.655462,0.533333,0.586609,0.677419,0.495798,0.259259,0.375000
0,Logistic_L2,0.562364,0.584906,0.544118,0.500000,0.588235,0.045455,0.083333,0.625644,0.633333,0.613852,0.580645,0.647059,0.300000,0.395604
3,Weighted_Logistic,0.560730,0.518868,0.549837,0.583333,0.516340,0.045161,0.083832,0.625644,0.573333,0.611819,0.677419,0.546218,0.280000,0.396226
2,SMOTE_Logistic_Exploratory,0.552015,0.518868,0.589869,0.666667,0.513072,0.050955,0.094675,0.623204,0.566667,0.583763,0.612903,0.554622,0.263889,0.368932



Original feature count before reduction: 17
Selected feature count after reduction: 9

Selected features in Reduced Logistic model:
- RECCHRGE
- OVERAGE
- ROAM
- DROPVCE
- UNANSVCE
- CUSTCARE
- MOU
- log_Usage_Ratio
- CUSTCARE_PER_MONTH

Saved:
- final_comparison_long.csv
- final_comparison_wide.csv
- reduced_model_selected_features.csv
